# Simple: Privacy Metrics with PAMOLA.CORE

**Goal**: Learn privacy metrics basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load original and transformed data from examples/data_examples/
- Calculate privacy metrics (DCR, NNDR, Uniqueness) using execute()
- Compare privacy risk between datasets
- Understand privacy assessment results

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Installation detection messages (✓ marks)
- Project root and working directory paths
- Import success confirmation
- Notebook start timestamp

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── metrics/
        └── 01_privacy_metrics_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.metrics.operations.privacy_ops import PrivacyMetricOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load original and transformed datasets
- Auto-creates sample data if files missing
- Review both dataset structures

**What you'll see:**
- Two dataset summaries (100 records each, 7 columns)
- First 5 records from original and transformed data
- Column statistics (unique values, ranges)
- Privacy-preserving transformations (noise, generalization)

**Note:** Transformed data includes noise and generalization to simulate anonymization while preserving statistical properties

In [ ]:
examples_dir = project_root / 'examples'
original_path = examples_dir / 'data_examples' / 'sample.csv'
transformed_path = examples_dir / 'data_examples' / 'sample.csv'

# Create sample original data if needed
if not original_path.exists():
    print("⚠️  Original file not found, creating sample data...")
    original_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create original data with sensitive information
    np.random.seed(42)
    original_data = pd.DataFrame({
        'record_id': range(1, 101),
        'age': np.random.randint(18, 80, 100),
        'income': np.random.randint(20000, 150000, 100),
        'education': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], 100),
        'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'], 100),
        'zip_code': np.random.randint(10000, 99999, 100),
        'health_score': np.random.uniform(50, 100, 100).round(2)
    })
    original_data.to_csv(original_path, index=False)
    print(f"✓ Original data created")

# Create sample transformed data if needed
if not transformed_path.exists():
    print("⚠️  Transformed file not found, creating sample data...")
    
    # Load original if just created
    original_data = pd.read_csv(original_path)
    
    # Create transformed data with added noise for privacy
    np.random.seed(123)
    transformed_data = original_data.copy()
    
    # Add noise to numeric fields
    transformed_data['age'] = transformed_data['age'] + np.random.randint(-3, 4, len(transformed_data))
    transformed_data['income'] = transformed_data['income'] + np.random.randint(-5000, 5001, len(transformed_data))
    transformed_data['health_score'] = (transformed_data['health_score'] + np.random.uniform(-5, 5, len(transformed_data))).clip(0, 100).round(2)
    
    # Generalize some categorical fields
    transformed_data['zip_code'] = (transformed_data['zip_code'] // 100) * 100
    
    transformed_data.to_csv(transformed_path, index=False)
    print(f"✓ Transformed data created")

# Load data
original_df = read_csv(original_path)
transformed_df = read_csv(transformed_path)

print(f"\n📊 Original Dataset: {len(original_df)} records, {len(original_df.columns)} columns")
print(f"📊 Transformed Dataset: {len(transformed_df)} records, {len(transformed_df.columns)} columns")

print(f"\n🔍 Original Data (First 5 records):")
print(original_df.head())

print(f"\n🔍 Transformed Data (First 5 records):")
print(transformed_df.head())

print(f"\n📈 Column Statistics:")
for col in original_df.columns:
    dtype_str = str(original_df[col].dtype)
    if pd.api.types.is_string_dtype(original_df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {original_df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={original_df[col].min()}, max={original_df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE configuration** in `create_config_kwargs()` function
   - Modify privacy metrics, quasi-identifiers, and parameters
2. Run to validate configuration and setup environment

**What you'll see:**
- Configuration validation (metrics and parameters)
- Parameter details for each metric (DCR, NNDR, uniqueness)
- Task directory created (✓)
- TaskReporter, DataSource (2 datasets), ProgressTracker initialized (✓)

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "original_dataset_name": "original_dataset",
        "transformed_dataset_name": "transformed_dataset",
        "privacy_metrics": ["dcr", "nndr", "uniqueness"],
        "metric_params": {
            "dcr": {"distance_metric": "euclidean"},
            "uniqueness": {
                "quasi_identifiers": ["age", "zip_code"],
                "sensitives": ["health_score"]
            }
        }
    }
```

**Note:** Quasi-identifiers and sensitive attributes must be specified for uniqueness metrics

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "original_dataset_name": "original_dataset",
        "transformed_dataset_name": "transformed_dataset",
        "privacy_metrics": ["dcr", "nndr", "uniqueness"],
        "metric_params": {
            "dcr": {
                "distance_metric": "euclidean",
                "normalize_features": True,
                "percentiles": [5, 10, 15, 20]
            },
            "nndr": {
                "distance_metric": "euclidean",
                "normalize_features": True,
                "threshold": 0.5
            },
            "uniqueness": {
                "quasi_identifiers": ["resume_id"],
                "sensitives": [],
                "k_values": [1],
                "l_diversity": False,
                "t_closeness": False
            }
        },
        "columns": ["years_of_experience", "current_salary_cad", "resume_id"]
    }

kwargs = create_config_kwargs()
privacy_metrics = kwargs["privacy_metrics"]
metric_params = kwargs["metric_params"]
columns = kwargs["columns"]
original_dataset_name = kwargs['original_dataset_name']
transformed_dataset_name = kwargs['transformed_dataset_name']

print(f"\n🔍 Validating configuration...\n")
print(f"✓ Privacy metrics to calculate: {', '.join(privacy_metrics)}")
for metric, params in metric_params.items():
    print(f"  {metric}:")
    for key, value in params.items():
        print(f"    - {key}: {value}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="privacy_assessment",
    description="Privacy metrics calculation for anonymized data.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {
    "original_dataset_name": original_dataset_name,
    "transformed_dataset_name": transformed_dataset_name
}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource with both datasets
data_source = DataSource(dataframes={
    "original_dataset": original_df,
    "transformed_dataset": transformed_df
})
print("✓ DataSource created with original and transformed data")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description="Processing privacy metrics",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration parameters below
- Run to execute privacy metrics calculation
- Monitor progress bar (6 processing steps)

**What you'll see:**
- Configuration summary (metrics, columns, parameters)
- Progress bar: validation → calculation → metrics → viz → save
- Completion status with artifact count (2-4 files)
- Status = "completed" (verify no errors)

**Key parameters:**
- `privacy_metrics=['dcr', 'nndr', 'uniqueness']`: Metrics to calculate
- `metric_params`: Custom parameters per metric
- `normalize=True`: Normalize features before distance calculations
- `generate_visualization=True`: Create charts

**Privacy Metrics:**
- **DCR (Distance to Closest Record)**: Measures re-identification risk via nearest original record
- **NNDR (Nearest Neighbor Distance Ratio)**: Assesses uniqueness via neighbor distance ratios
- **Uniqueness**: K-anonymity, l-diversity, t-closeness for group-based privacy

**Note:** Higher DCR/NNDR scores indicate better privacy protection

In [ ]:
# Create operation with production-style configuration
operation = PrivacyMetricOperation(
    # Core parameters
    privacy_metrics=privacy_metrics,
    metric_params=metric_params,
    
    # Column selection
    columns=columns,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    normalize=True,
    sample_size=None,  # Use all data
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Privacy metrics:   {', '.join(operation.privacy_metrics)}")
print(f"  Columns:           {', '.join(operation.columns) if operation.columns else 'All'}")
print(f"  Normalize:         {operation.normalize}")
print(f"  Visualizations:    {operation.generate_visualization}")
print(f"  Save output:       {operation.save_output}")
print(f"  Force recalc:      {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing privacy metrics operation...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load and interpret privacy metrics
- Review risk assessment for each metric
- Assess overall privacy level

**What you'll see:**
- Generated artifacts list (metrics, visualizations)
- DCR statistics (mean, min, max, privacy score)
- NNDR statistics (ratios, high-risk record count)
- Uniqueness results (k-anonymity violations, l-diversity, t-closeness)
- Interpretations for each metric

**Generated artifacts:**
- Metrics JSON in metrics/
- Visualizations in visualizations/

**Note:** Lower DCR and k-anonymity violations indicate higher re-identification risk. Higher NNDR and privacy scores indicate better protection.

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load metrics file
metrics_files = list(task_dir.glob('metrics/*.json'))
# Filter out data_types files
metrics_files = [f for f in metrics_files if not f.name.startswith('data_types')]

if metrics_files:
    # Sort by modification time and get latest
    metrics_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    metrics_file = metrics_files[0]
    
    with open(metrics_file, 'r') as f:
        metrics_data = json.load(f)
    
    metrics_results = metrics_data.get('metrics', {})
    
    print("\n" + "=" * 80)
    print("📊 PRIVACY METRICS RESULTS")
    print("=" * 80)
    
    # DCR Results
    if 'dcr' in metrics_results:
        dcr = metrics_results['dcr']
        print("\n🔒 DCR (Distance to Closest Record):")
        if 'dcr_statistics' in dcr:
            stats = dcr['dcr_statistics']
            print(f"  Mean distance:     {stats.get('mean', 0):.4f}")
            print(f"  Min distance:      {stats.get('min', 0):.4f}")
            print(f"  Max distance:      {stats.get('max', 0):.4f}")
            print(f"  Std deviation:     {stats.get('std', 0):.4f}")
        if 'privacy_score' in dcr:
            print(f"  Privacy score:     {dcr['privacy_score']:.4f} (0-1, higher=better)")
        if 'interpretation' in dcr:
            print(f"  \n  Interpretation: {dcr['interpretation']}")
    
    # NNDR Results
    if 'nndr' in metrics_results:
        nndr = metrics_results['nndr']
        print("\n🔒 NNDR (Nearest Neighbor Distance Ratio):")
        if 'nndr_statistics' in nndr:
            stats = nndr['nndr_statistics']
            print(f"  Mean ratio:        {stats.get('mean', 0):.4f}")
            print(f"  Min ratio:         {stats.get('min', 0):.4f}")
            print(f"  Max ratio:         {stats.get('max', 0):.4f}")
        if 'high_risk_records' in nndr:
            print(f"  High risk records: {nndr['high_risk_records']}")
        if 'interpretation' in nndr:
            print(f"  \n  Interpretation: {nndr['interpretation']}")
    
    # Uniqueness Results
    if 'uniqueness' in metrics_results:
        uniq = metrics_results['uniqueness']
        print("\n🔒 Uniqueness Metrics:")
        
        if 'k_anonymity' in uniq:
            k_anon = uniq['k_anonymity']
            print(f"\n  K-Anonymity:")
            print(f"    Identity disclosure: {k_anon.get('identity_disclosure_rate', 0):.2%}")
            print(f"    Number of groups:    {k_anon.get('num_groups', 0)}")
            if 'k_anonymity_stats' in k_anon:
                print(f"    \n    K-value violations:")
                for stat in k_anon['k_anonymity_stats']:
                    k_val = stat['k_value']
                    violations = stat['num_violations']
                    pct = stat['percent_violation']
                    print(f"      k={k_val}: {violations} violations ({pct:.1f}%)")
        
        if 'l_diversity' in uniq:
            l_div = uniq['l_diversity']
            print(f"\n  L-Diversity:")
            print(f"    Min diversity:  {l_div.get('min_l_diversity', 0)}")
            print(f"    Max diversity:  {l_div.get('max_l_diversity', 0)}")
            print(f"    Avg diversity:  {l_div.get('avg_l_diversity', 0):.2f}")
        
        if 't_closeness' in uniq:
            t_close = uniq['t_closeness']
            print(f"\n  T-Closeness:")
            print(f"    Min t-closeness: {t_close.get('min_t_closeness', 0):.4f}")
            print(f"    Max t-closeness: {t_close.get('max_t_closeness', 0):.4f}")
            print(f"    Avg t-closeness: {t_close.get('avg_t_closeness', 0):.4f}")
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Metrics calculated: {len(metrics_results)}")
    print(f"  Privacy assessment complete!")
    print(f"\n💡 Higher DCR/NNDR scores = Better privacy protection!")
else:
    print("⚠️  No metrics file found in", task_dir / 'metrics')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**What you'll see:**
- Directory structure with file counts
- File names with sizes (KB) for each artifact
- Up to 5 files listed per directory

**Output structure:**
```
examples/data_examples/simple_output/
├── metrics/          # JSON metrics
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Privacy metrics with detailed statistics
2. **Visualizations**: Charts showing privacy risk distributions

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / "metrics"

if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")

else:
    metrics_files = list(metrics_dir.glob("*.json"))

    if not metrics_files:
        print("⚠️  No metrics files found")

    else:
        # 1) Exclude data_types*
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]

        print(f"\n📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB")
        print("=" * 80)

        try:
            with open(latest_metrics_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            metadata = data.get("metadata", {})
            metrics = data.get("metrics", {})

            # METADATA
            print("\n📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if "operation" in metadata:
                op = metadata["operation"]
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")

            # DCR
            if "dcr" in metrics:
                dcr = metrics["dcr"]
                print("\n📊 DCR (DISTANCE TO CLOSEST RECORD):")
                print("=" * 60)

                if "dcr_statistics" in dcr:
                    print("\n   Statistics:")
                    for k, v in dcr["dcr_statistics"].items():
                        if isinstance(v, float):
                            print(f"      {k:15s}: {v:.8f}")
                        else:
                            print(f"      {k:15s}: {v}")

                if "risk_assessment" in dcr:
                    print("\n   Risk Assessment:")
                    for k, v in dcr["risk_assessment"].items():
                        print(f"      {k:15s}: {v}")

                if "proportion_at_risk" in dcr:
                    print(f"\n   Proportion At Risk: {dcr['proportion_at_risk']:.2%}")

                if "privacy_score" in dcr:
                    print(f"   Privacy Score: {dcr['privacy_score']:.8f} (0–1, higher is better)")

                if "interpretation" in dcr:
                    print("\n   Interpretation:")
                    print(f"      {dcr['interpretation']}")

            # NNDR
            if "nndr" in metrics:
                nndr = metrics["nndr"]
                print("\n\n📊 NNDR (NEAREST NEIGHBOR DISTANCE RATIO):")
                print("=" * 60)

                if "nndr_statistics" in nndr:
                    print("\n   Statistics:")
                    for k, v in nndr["nndr_statistics"].items():
                        if isinstance(v, float):
                            print(f"      {k:15s}: {v:.6f}")
                        else:
                            print(f"      {k:15s}: {v}")

                print(f"\n   High Risk Records: {nndr.get('high_risk_records', 0)}")

                if "high_risk_proportion" in nndr:
                    print(f"   High Risk Proportion: {nndr['high_risk_proportion']:.2%}")

                if "privacy_classification" in nndr:
                    print("\n   Privacy Classification:")
                    for k, v in nndr["privacy_classification"].items():
                        print(f"      {k:15s}: {v}")

                if "interpretation" in nndr:
                    print("\n   Interpretation:")
                    print(f"      {nndr['interpretation']}")

            # UNIQUENESS
            if "uniqueness" in metrics:
                uniq = metrics["uniqueness"]
                print("\n\n📊 UNIQUENESS METRICS:")
                print("=" * 60)

                # K-anonymity
                if "k_anonymity" in uniq:
                    k_anon = uniq["k_anonymity"]
                    print("\n   K-Anonymity:")
                    print(f"      Identity Disclosure Rate: {k_anon.get('identity_disclosure_rate', 0):.4f}")
                    print(f"      Number of Groups: {k_anon.get('num_groups', 0)}")

                    if "k_anonymity_stats" in k_anon:
                        print("\n      Violations:")
                        print(f"         {'K':>6} {'Violations':>12} {'Percent':>10}")
                        print("         " + "-" * 32)
                        for s in k_anon["k_anonymity_stats"]:
                            print(
                                f"         {s['k_value']:>6} "
                                f"{s['num_violations']:>12} "
                                f"{s['percent_violation']:>9.2f}%"
                            )

                # L-diversity
                if "l_diversity" in uniq:
                    print("\n   L-Diversity:")
                    if not uniq["l_diversity"]:
                        print("      N/A (not computed)")
                    else:
                        ld = uniq["l_diversity"]
                        print(f"      Min: {ld.get('min_l_diversity')}")
                        print(f"      Max: {ld.get('max_l_diversity')}")
                        print(f"      Avg: {ld.get('avg_l_diversity'):.4f}")

                # T-closeness
                if "t_closeness" in uniq:
                    print("\n   T-Closeness:")
                    if not uniq["t_closeness"]:
                        print("      N/A (not computed)")
                    else:
                        tc = uniq["t_closeness"]
                        print(f"      Method: {tc.get('calculate_method', 'N/A')}")
                        print(f"      Min: {tc.get('min_t_closeness'):.6f}")
                        print(f"      Max: {tc.get('max_t_closeness'):.6f}")
                        print(f"      Avg: {tc.get('avg_t_closeness'):.6f}")

            # PERFORMANCE
            print("\n\n⚡ PERFORMANCE:")
            print("=" * 60)
            print(f"   Duration: {metrics.get('duration_seconds', 0):.4f}s")
            print(f"   Records Processed: {metrics.get('records_processed', 0):,}")
            print(f"   Records/Second: {metrics.get('records_per_second', 0):.2f}")

            if "sample_size" in metrics:
                if metrics["sample_size"] is None:
                    print("   Sample Size: N/A")
                else:
                    print(f"   Sample Size: {metrics['sample_size']:,}")

        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. VISUALIZATIONS (NEWEST BATCH)
print("\n\n2️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**  

✅ Load original and transformed data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure privacy metrics with full parameters  
✅ Execute using operation.execute()  
✅ Analyze privacy risk and review artifacts  

**Key takeaways:**
- **DCR (Distance to Closest Record)**: Measures how close synthetic records are to original data
  - Lower DCR = Higher privacy risk
  - Privacy score ranges from 0-1 (higher is better)
- **NNDR (Nearest Neighbor Distance Ratio)**: Compares distances to 1st and 2nd nearest neighbors
  - Values close to 1 indicate realistic synthetic data
  - Values close to 0 indicate high re-identification risk
- **Uniqueness Metrics**: Assess re-identification risks
  - K-anonymity: Each record indistinguishable from k-1 others
  - L-diversity: Diverse sensitive attribute values per group
  - T-closeness: Similar distribution within groups and population
- Multiple metrics provide comprehensive privacy assessment
- Visualizations help identify privacy vulnerabilities

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_privacy_metrics_advanced.ipynb`** includes:
- Custom distance metrics and normalization strategies
- Advanced k-anonymity configurations with multiple quasi-identifiers
- Comparative analysis across different anonymization techniques
- Large-scale processing with FAISS acceleration
- Privacy risk mitigation strategies
- Integration with profiling results

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 📊 [Privacy Metrics Guide](../../docs/privacy_metrics.md)